### Setup


In [1]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [2]:
import tomllib
import numpy as np
import bambi as bmb
import arviz as az
from scipy.special import expit

from prettytable import PrettyTable

from src.stat_utils import *
from src.anl_utils import load_data, get_session_data

In [12]:
def one_sided_posterior_prob(idata, predictor, direction='greater'):
    """Calculate the one-sided posterior probability that the samples are greater than zero."""
    
    beta = idata.posterior[predictor].values.flatten()
    if direction == 'greater':
        prob = np.mean(beta > 0)
    elif direction == 'less':
        prob = np.mean(beta < 0)
    else:
        raise ValueError("Direction must be 'greater' or 'less'.")
    return prob

def odds_ratio_summary(idata, predictor):
    """Calculate the odds ratio summary statistics for a given predictor."""
    beta = idata.posterior[predictor].values.flatten()
    or_mean = np.exp(beta).mean()
    or_low = np.percentile(np.exp(beta), 2.5)
    or_high = np.percentile(np.exp(beta), 97.5)
    return or_mean, or_low, or_high

def posterior_table(idata, predictors, directions):
    """Summarize the model by calculating posterior probabilities for each predictor."""
    table = PrettyTable()
    table.field_names = ["Predictor", "direction", "P"]
    
    for predictor, direction in zip(predictors, directions):
        prob = one_sided_posterior_prob(idata, predictor, direction)
        table.add_row([predictor, direction, f"{prob:.3f}"])
    
    return table

def OR_table(idata, predictors):
    """Summarize the model by calculating posterior probabilities and odds ratios for each predictor."""
    table = PrettyTable()
    table.field_names = ["Predictor", "Mean", "Lower (2.5%)", "Upper (97.5%)"]
    
    for predictor in predictors:
        or_mean, or_low, or_high = odds_ratio_summary(idata, predictor)
        table.add_row([predictor, f"{or_mean:.3f}", f"{or_low:.3f}", f"{or_high:.3f}"])
    
    return table

### Load and prepare data

In [3]:
sim_results_folder = '../results/simulation'
data_folder = '../data'

sync_at_file = os.path.join(sim_results_folder, 'first_session_arnold_tongues.npy')
fr_at_file = os.path.join(sim_results_folder, 'first_session_firing_rates.npy')
emp_at_file = os.path.join(data_folder, 'Experiment.csv')

In [4]:
# Load the simulations results and the empirical data
sync_results = np.load(sync_at_file)
sync_results_vector = sync_results.mean(axis=0).flatten()


#fr_results_vector = fr_results.mean(axis=0).flatten()
fr_results = np.load(fr_at_file)
fr_results = fr_results.mean(axis=0)
fr_results -= np.outer(fr_results[:,-1], np.ones(5))
fr_results_vector = fr_results.flatten()

data = load_data(emp_at_file)
# Get the data for the first session
data = get_session_data(data, 1)

# Map synchrony values to each condition in DataFrame
data['Synchrony'] = data['Condition'].apply(lambda x: sync_results_vector[x-1])
data['FiringRate'] = data['Condition'].apply(lambda x: fr_results_vector[x-1])

data = zscore_data(data, ['ContrastHeterogeneity', 'GridCoarseness', 'Synchrony', 'FiringRate'])

In [ ]:
# Features-only hierarchical logistic regression
model_features = bmb.Model(
    "Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

# Model synchrony (mechanism)
model_sync = bmb.Model(
    "Correct ~ 1 + Synchrony + (1 + Synchrony | SubjectID)",
    data=data,
    family="bernoulli"
)

# Full model (features + mechanism)
model_sync_full = bmb.Model(
    "Correct ~ 1 + Synchrony + ContrastHeterogeneity * GridCoarseness + (1 + Synchrony + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)


# Firing rate control model
model_fr = bmb.Model(
    "Correct ~ 1 + FiringRate + (1 + FiringRate | SubjectID)",
    data=data,
    family="bernoulli"
)

# Full model (firing rate instead of synchrony)
model_fr_full = bmb.Model(
    "Correct ~ 1 + FiringRate + ContrastHeterogeneity * GridCoarseness + (1 + FiringRate + ContrastHeterogeneity * GridCoarseness|SubjectID)",
    data=data,
    family="bernoulli"
)

# Proper full

In [29]:
# To print the priors for each term
model_features.build()
print(model_features)

model_sync.build()
print(model_sync)

model_full.build()
print(model_full)


       Formula: Correct ~ 1 + ContrastHeterogeneity * GridCoarseness + (1 + ContrastHeterogeneity * GridCoarseness|SubjectID)
        Family: bernoulli
          Link: p = logit
  Observations: 6000
        Priors: 
    target = p
        Common-level effects
            Intercept ~ Normal(mu: 0.0, sigma: 2.5)
            ContrastHeterogeneity ~ Normal(mu: 0.0, sigma: 2.5000000000000004)
            GridCoarseness ~ Normal(mu: 0.0, sigma: 2.5000000000000004)
            ContrastHeterogeneity:GridCoarseness ~ Normal(mu: 0.0, sigma: 2.5000000000000004)
        
        Group-level effects
            1|SubjectID ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 2.5))
            ContrastHeterogeneity|SubjectID ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 2.5000000000000004))
            GridCoarseness|SubjectID ~ Normal(mu: 0.0, sigma: HalfNormal(sigma: 2.5000000000000004))
            ContrastHeterogeneity:GridCoarseness|SubjectID ~ Normal(mu: 0.0, sigma: HalfNormal(sigma:
                2.5000

Are the factors that determine synchrony among coupled oscillators (frequency detuning and coupling strength) predictive of human ability to segregate a rectangular figure from its background in texture stimuli? 

In [ ]:
idata_features = model_features.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)


Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]


/home/mario/miniconda3/envs/bat_env/lib/python3.13/site-packages/rich/live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 243 seconds.


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.712,0.183,0.360,1.094,0.004,0.004,2427.0,2849.0,1.0
ContrastHeterogeneity,-0.596,0.154,-0.896,-0.288,0.003,0.003,2475.0,2526.0,1.0
GridCoarseness,-0.268,0.070,-0.402,-0.125,0.001,0.002,4165.0,4167.0,1.0
ContrastHeterogeneity:GridCoarseness,0.233,0.060,0.108,0.346,0.001,0.001,4362.0,3972.0,1.0


In [13]:
predictors = ["ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness"]
directions = ['less', 'less', 'greater']

posterior = posterior_table(idata_features, predictors, directions)
odds_ratios = OR_table(idata_features, predictors)
print(posterior)
print(odds_ratios)

az.summary(idata_features, var_names=predictors, hdi_prob=0.95)

+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|        ContrastHeterogeneity         |    less   | 0.997 |
|            GridCoarseness            |    less   | 0.999 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 0.999 |
+--------------------------------------+-----------+-------+
+--------------------------------------+-------+--------------+---------------+
|              Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+-------+--------------+---------------+
|        ContrastHeterogeneity         | 0.558 |    0.409     |     0.751     |
|            GridCoarseness            | 0.767 |    0.665     |     0.878     |
| ContrastHeterogeneity:GridCoarseness | 1.265 |    1.125     |     1.430     |
+--------------------------------------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ContrastHeterogeneity,-0.596,0.154,-0.896,-0.288,0.003,0.003,2475.0,2526.0,1.0
GridCoarseness,-0.268,0.070,-0.402,-0.125,0.001,0.002,4165.0,4167.0,1.0
ContrastHeterogeneity:GridCoarseness,0.233,0.060,0.108,0.346,0.001,0.001,4362.0,3972.0,1.0


In [32]:
az.summary(idata_features, var_names=["sd", "r"], filter_vars="like")

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.712,0.183,0.366,1.056,0.004,0.004,2427.0,2849.0,1.0
ContrastHeterogeneity,-0.596,0.154,-0.871,-0.294,0.003,0.003,2475.0,2526.0,1.0
GridCoarseness,-0.268,0.070,-0.404,-0.144,0.001,0.002,4165.0,4167.0,1.0
ContrastHeterogeneity:GridCoarseness,0.233,0.060,0.121,0.345,0.001,0.001,4362.0,3972.0,1.0
ContrastHeterogeneity|SubjectID_sigma,0.392,0.144,0.181,0.647,0.003,0.005,2734.0,3567.0,1.0
GridCoarseness|SubjectID_sigma,0.157,0.074,0.043,0.293,0.001,0.002,2172.0,2689.0,1.0
ContrastHeterogeneity:GridCoarseness|SubjectID_sigma,0.135,0.072,0.012,0.262,0.001,0.001,2232.0,2644.0,1.0
ContrastHeterogeneity|SubjectID[1],-0.042,0.168,-0.368,0.273,0.003,0.003,2885.0,2649.0,1.0
ContrastHeterogeneity|SubjectID[2],0.134,0.166,-0.176,0.448,0.003,0.003,2817.0,3237.0,1.0
ContrastHeterogeneity|SubjectID[3],-0.491,0.180,-0.830,-0.159,0.003,0.003,2986.0,3110.0,1.0


Does the synchronization behavior of a biophysical model of V1 predict human ability to segregate a rectangular figure from its background in texture stimuli?

In [14]:
idata_sync = model_sync.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 196 seconds.


In [ ]:
predictors = ["Synchrony"]
directions = ['greater']

posterior = posterior_table(idata_sync, predictors, directions)
odds_ratios = OR_table(idata_sync, predictors)
print(posterior)
print(odds_ratios)

az.summary(idata_sync, var_names=predictors, hdi_prob=0.95)

+-----------+-----------+-------+
| Predictor | direction |   P   |
+-----------+-----------+-------+
| Synchrony |  greater  | 0.998 |
+-----------+-----------+-------+
+-----------+-------+--------------+---------------+
| Predictor |  Mean | Lower (2.5%) | Upper (97.5%) |
+-----------+-------+--------------+---------------+
| Synchrony | 2.198 |    1.389     |     3.407     |
+-----------+-------+--------------+---------------+


,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Synchrony,0.764,0.219,0.324,1.216,0.004,0.004,2485.0,2776.0,1.0


In [31]:
az.summary(idata_sync, var_names=["sd", "r"], filter_vars="like")

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.745,0.206,0.343,1.123,0.004,0.004,2246.0,3035.0,1.0
Synchrony,0.764,0.219,0.348,1.188,0.004,0.004,2485.0,2776.0,1.0
Synchrony|SubjectID_sigma,0.579,0.219,0.269,0.963,0.005,0.010,2125.0,3510.0,1.0
Synchrony|SubjectID[1],0.077,0.241,-0.397,0.522,0.004,0.004,2942.0,3595.0,1.0
Synchrony|SubjectID[2],-0.279,0.232,-0.734,0.148,0.004,0.004,2733.0,3224.0,1.0
Synchrony|SubjectID[3],0.829,0.293,0.287,1.382,0.005,0.003,4078.0,5050.0,1.0
Synchrony|SubjectID[4],-0.693,0.228,-1.125,-0.261,0.004,0.004,2646.0,2662.0,1.0
Synchrony|SubjectID[5],0.067,0.242,-0.397,0.531,0.004,0.004,2975.0,3192.0,1.0
Synchrony|SubjectID[6],-0.018,0.239,-0.476,0.431,0.004,0.004,2827.0,3440.0,1.0
Synchrony|SubjectID[7],0.349,0.255,-0.141,0.827,0.004,0.004,3275.0,4123.0,1.0


Does model synchrony add predictive power beyond the experimentally manipulated stimulus features? 

In [18]:
idata_full = model_full.fit(
    draws=2000, tune=2000, target_accept=0.95,
    idata_kwargs={"log_likelihood": True}, progressbar=False
)

Modeling the probability that Correct==1
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [Intercept, Synchrony, ContrastHeterogeneity, GridCoarseness, ContrastHeterogeneity:GridCoarseness, 1|SubjectID_sigma, 1|SubjectID_offset, Synchrony|SubjectID_sigma, Synchrony|SubjectID_offset, ContrastHeterogeneity|SubjectID_sigma, ContrastHeterogeneity|SubjectID_offset, GridCoarseness|SubjectID_sigma, GridCoarseness|SubjectID_offset, ContrastHeterogeneity:GridCoarseness|SubjectID_sigma, ContrastHeterogeneity:GridCoarseness|SubjectID_offset]
Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 298 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


In [25]:
# Compare models (LOO)
az.compare({
    "stimulus features": idata_features,
    "synchrony": idata_sync,
    "full model": idata_full,
}, method="BB-pseudo-BMA")

,rank,elpd_loo,p_loo,elpd_diff,weight,se,dse,warning,scale
full model,0,-3581.863886,32.371590,0.000000,6.377352e-01,29.259614,0.000000,False,log
stimulus features,1,-3582.934219,28.352039,1.070333,3.622648e-01,29.187150,3.048846,False,log
synchrony,2,-3632.606110,16.723408,50.742224,3.055780e-10,28.081396,10.877060,False,log


In [21]:
predictors = ["ContrastHeterogeneity", "GridCoarseness", "ContrastHeterogeneity:GridCoarseness", "Synchrony"]
directions = ['less', 'less', 'greater', 'greater']

posterior = posterior_table(idata_full, predictors, directions)
odds_ratios = OR_table(idata_full, predictors)
print(posterior)
print(odds_ratios)

az.summary(idata_full, var_names=predictors, hdi_prob=0.95)

+--------------------------------------+-----------+-------+
|              Predictor               | direction |   P   |
+--------------------------------------+-----------+-------+
|        ContrastHeterogeneity         |    less   | 0.998 |
|            GridCoarseness            |    less   | 0.998 |
| ContrastHeterogeneity:GridCoarseness |  greater  | 0.996 |
|              Synchrony               |  greater  | 0.872 |
+--------------------------------------+-----------+-------+
+--------------------------------------+-------+--------------+---------------+
|              Predictor               |  Mean | Lower (2.5%) | Upper (97.5%) |
+--------------------------------------+-------+--------------+---------------+
|        ContrastHeterogeneity         | 0.603 |    0.464     |     0.786     |
|            GridCoarseness            | 0.791 |    0.693     |     0.897     |
| ContrastHeterogeneity:GridCoarseness | 1.227 |    1.104     |     1.358     |
|              Synchrony        

,mean,sd,hdi_2.5%,hdi_97.5%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
ContrastHeterogeneity,-0.514,0.130,-0.776,-0.253,0.002,0.002,3881.0,3543.0,1.0
GridCoarseness,-0.237,0.064,-0.363,-0.105,0.001,0.001,5195.0,4530.0,1.0
ContrastHeterogeneity:GridCoarseness,0.203,0.056,0.096,0.303,0.001,0.004,5794.0,3519.0,1.0
Synchrony,0.143,0.137,-0.132,0.413,0.002,0.002,5112.0,4740.0,1.0


In [30]:
az.summary(idata_full, var_names=["sd", "r"], filter_vars="like")

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
Intercept,0.729,0.190,0.358,1.077,0.004,0.003,2638.0,3840.0,1.0
Synchrony,0.143,0.137,-0.105,0.411,0.002,0.002,5112.0,4740.0,1.0
ContrastHeterogeneity,-0.514,0.130,-0.748,-0.253,0.002,0.002,3881.0,3543.0,1.0
GridCoarseness,-0.237,0.064,-0.358,-0.116,0.001,0.001,5195.0,4530.0,1.0
ContrastHeterogeneity:GridCoarseness,0.203,0.056,0.108,0.303,0.001,0.004,5794.0,3519.0,1.0
Synchrony|SubjectID_sigma,0.298,0.145,0.049,0.575,0.002,0.002,3109.0,2769.0,1.0
ContrastHeterogeneity|SubjectID_sigma,0.311,0.130,0.114,0.545,0.002,0.003,4048.0,5208.0,1.0
GridCoarseness|SubjectID_sigma,0.135,0.072,0.005,0.259,0.001,0.001,2277.0,2567.0,1.0
ContrastHeterogeneity:GridCoarseness|SubjectID_sigma,0.087,0.066,0.000,0.193,0.002,0.005,2189.0,2765.0,1.0
Synchrony|SubjectID[1],0.146,0.178,-0.173,0.496,0.002,0.002,5878.0,6174.0,1.0


### Sensitivity Analysis

In [ ]:
with open('../config/simulation/simulation.toml', 'rb') as f:
    sim_config = tomllib.load(f)

seed = sim_config['random_seed']
rng = np.random.default_rng(seed)

In [ ]:
num_repetitions = 100
effect_sizes = np.linspace(0.3, 0.9, 7) # log-odds

formula = "Correct ~ 1 + Synchrony + (1|SubjectID) + (0 + Synchrony|SubjectID)"
effect_of_interest = "Synchrony"

In [ ]:
subject_index, num_subjects = create_subject_index(data)
synchrony = data["Synchrony"].to_numpy()

# Extract posteriors for intercept and beta_sync as well as for the subject-level random effects
posteriors = az.extract(idata_pure, combined=True)

intercept = np.median(posteriors["Intercept"].to_numpy())
beta_sync = np.median(posteriors["Synchrony"].to_numpy())

sdev_intercept = np.median(posteriors["1|SubjectID_sigma"].to_numpy())              # random intercept SD
sdev_beta_sync = np.median(posteriors["Synchrony|SubjectID_sigma"].to_numpy())    # random slope SD


results = []
table = PrettyTable()
table.field_names = ["Effect Size (log-odds)", "Detection Rate"]
for effect_size in effect_sizes:
    detections = 0
    for r in range(num_repetitions):
        simulated_df = simulate_correct(data, effect_size, synchrony, intercept, sdev_beta_sync, sdev_intercept, subject_index, num_subjects, rng)
        detected = fit_and_decide(simulated_df, formula, effect_of_interest, draws=1000,
                   tune=1000, threshold=0.975)
        detections += int(detected)
    detection_rate = detections / num_repetitions
    results.append({"true_beta_sync": effect_size, "detect_rate": detection_rate})
    table.add_row([effect_size, detection_rate])

print(table)
